# Stress Prediction V36 — CONFIG 기반 Ablation (v34 로직 복원)

V35에서 4가지를 한 번에 바꿔서 Tree_MAE가 0.127 -> 0.18로 크게 나빠졌습니다.
원인은 십중팔구 **v34의 "개별 트리 예측값들의 0.54분위수" 집계 트릭을 표준 평균(predict)으로
바꿔버린 것**입니다. `max_features=1, min_samples_leaf=1`로 극단 랜덤화된 트리들은
노이즈가 커서, 34번 튜닝 끝에 찾은 quantile 집계가 그 노이즈를 상쇄해주고 있었을 가능성이 높습니다.

이 노트북은:
1. **`USE_TREE_QUANTILE_TRICK=True`가 기본값** — v34 핵심 로직을 그대로 복원 (안전한 기준점)
2. 나머지 새 요소(도메인 피처, allostatic load, absolute_error, pair k-NN, SVR, 스태킹)는
   `CONFIG`에서 **하나씩** 켜고 꺼가며 v34 기준 CV MAE(약 0.127)와 비교할 수 있게 구성
3. 매 실행마다 "v34 원본 고정 블렌드(0.72*tree+0.28*pair)" 점수도 같이 출력해서 sanity check 가능

**사용법**: 아래 `CONFIG`에서 플래그 하나만 바꿔서 실행 -> CV MAE 확인 -> 원상복구 -> 다음 플래그 테스트,
이런 식으로 한 번에 하나씩만 바꿔가며 어떤 요소가 실제로 도움되는지 확인해주세요.


In [ ]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42

# ============== 여기서 하나씩만 바꿔가며 테스트하세요 ==============
CONFIG = dict(
    use_domain_features=False,      # bp_stage / glucose_stage / cholesterol_stage 추가 여부
    use_allostatic_load=False,      # allostatic_load 피처 추가 여부
    tree_criterion="squared_error", # v34 원본. "absolute_error"로 바꿔서 비교 가능
    use_tree_quantile_trick=True,   # v34 원본 핵심 로직 (False로 하면 표준 predict() 사용 - V35에서 문제였던 방식)
    tree_quantile=0.54,
    n_estimators=1200,              # v34 원본값. absolute_error 쓸 경우 800 정도로 낮춰도 됨
    use_pair_knn=False,             # v34 원본은 argmin 1개(False). True면 k=5 평균
    pair_k=5,
    pair_quantile=0.48,
    add_svr=False,                  # SVR을 3번째 모델로 추가할지 여부
    use_stacking=False,             # True면 RidgeCV 스태킹, False면 v34 고정 블렌드(0.72*tree+0.28*pair) 사용
)
# =====================================================================

N_ESTIMATORS = CONFIG["n_estimators"]

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]

ALLOSTATIC_COLUMNS = [
    "systolic_blood_pressure", "diastolic_blood_pressure",
    "cholesterol", "glucose", "bmi", "bone_density",
]


In [ ]:

def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """V34 원본과 동일 (행 단위, leakage 없음)."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def _bp_stage(sbp, dbp):
    if pd.isna(sbp) or pd.isna(dbp):
        return "unknown"
    if sbp >= 140 or dbp >= 90:
        return "stage2"
    if sbp >= 130 or dbp >= 80:
        return "stage1"
    if sbp >= 120:
        return "elevated"
    return "normal"


def _glucose_stage(g):
    if pd.isna(g):
        return "unknown"
    if g >= 126:
        return "diabetes_range"
    if g >= 100:
        return "prediabetes_range"
    return "normal"


def _cholesterol_stage(c):
    if pd.isna(c):
        return "unknown"
    if c >= 240:
        return "high"
    if c >= 200:
        return "borderline"
    return "desirable"


def add_domain_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["bp_stage"] = [
        _bp_stage(s, d)
        for s, d in zip(result["systolic_blood_pressure"], result["diastolic_blood_pressure"])
    ]
    result["glucose_stage"] = result["glucose"].apply(_glucose_stage)
    result["cholesterol_stage"] = result["cholesterol"].apply(_cholesterol_stage)
    return result


def compute_train_stats(train_frame, columns):
    return {
        col: (np.nanmean(train_frame[col].to_numpy(dtype=float)),
              np.nanstd(train_frame[col].to_numpy(dtype=float)) + 1e-6)
        for col in columns
    }


def add_allostatic_load(frame, train_stats):
    result = frame.copy()
    flags = pd.DataFrame(index=frame.index)
    for col, (mean, std) in train_stats.items():
        z = (frame[col] - mean) / std
        flags[f"{col}_abnormal"] = (z.abs() > 1).astype(int)
    result["allostatic_load"] = flags.sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def build_fold_features(train_raw: pd.DataFrame, other_raw: pd.DataFrame):
    train_feat = add_features(train_raw)
    other_feat = add_features(other_raw)
    if CONFIG["use_domain_features"]:
        train_feat = add_domain_features(train_feat)
        other_feat = add_domain_features(other_feat)
    if CONFIG["use_allostatic_load"]:
        stats = compute_train_stats(train_feat, ALLOSTATIC_COLUMNS)
        train_feat = add_allostatic_load(train_feat, stats)
        other_feat = add_allostatic_load(other_feat, stats)
    return train_feat, other_feat


In [ ]:

def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)

    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        criterion=CONFIG["tree_criterion"],
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(train_matrix, target)

    if CONFIG["use_tree_quantile_trick"]:
        # V34 원본 핵심 로직: 개별 트리 예측값들의 분위수
        tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
        return np.quantile(tree_predictions, CONFIG["tree_quantile"], axis=1)
    return model.predict(val_matrix)


def empirical_rank_transform(train_values: np.ndarray, test_values: np.ndarray):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        if CONFIG["use_pair_knn"]:
            k = min(CONFIG["pair_k"], len(target_values))
            nearest_idx = np.argpartition(distance, kth=k - 1, axis=1)[:, :k]
            neighbor_targets.append(target_values[nearest_idx].mean(axis=1))
        else:
            nearest_idx = np.argmin(distance, axis=1)
            neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), CONFIG["pair_quantile"], axis=1)


def svr_prediction(train_features, val_features, target) -> np.ndarray:
    preprocessor = make_preprocessor(train_features)
    train_matrix = preprocessor.fit_transform(train_features)
    val_matrix = preprocessor.transform(val_features)
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_matrix)
    val_scaled = scaler.transform(val_matrix)
    model = SVR(kernel="rbf", C=10.0, epsilon=0.01, gamma="scale")
    model.fit(train_scaled, target)
    return model.predict(val_scaled)


def blend(tree_pred, pair_pred, svr_pred=None):
    if CONFIG["add_svr"] and svr_pred is not None:
        combined = 0.6 * tree_pred + 0.25 * pair_pred + 0.15 * svr_pred
    else:
        combined = 0.72 * tree_pred + 0.28 * pair_pred  # v34 원본 고정 블렌드
    return np.clip(np.round(combined, 2), 0.0, 1.0)


## 5-Fold CV — CONFIG 설정대로 실행

`v34_reference_blend`는 CONFIG와 무관하게 항상 "0.72*tree+0.28*pair 고정 블렌드"로 계산해서,
지금 CONFIG 설정이 v34 기준 대비 실제로 좋아졌는지/나빠졌는지 매 실행마다 비교할 수 있게 했습니다.


In [ ]:

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")   # train/test/sample_submission이 있는 폴더
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")  # 제출 파일 저장 폴더
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

raw_X = train.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
raw_X_test = test.drop(columns=[ID_COLUMN]).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
oof_tree = np.zeros(len(y))
oof_pair = np.zeros(len(y))
oof_svr = np.zeros(len(y))
oof_reference = np.zeros(len(y))  # CONFIG와 무관한 v34 고정 블렌드 (sanity check용)
fold_rows = []

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_raw_fold = raw_X.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train_fold = y.iloc[train_idx].reset_index(drop=True)
    y_val_fold = y.iloc[val_idx].reset_index(drop=True)

    train_feat, val_feat = build_fold_features(train_raw_fold, val_raw_fold)

    pred_tree = tree_prediction(train_feat, val_feat, y_train_fold)
    pred_pair = pair_prediction(train_feat, val_feat, y_train_fold)
    pred_svr = svr_prediction(train_feat, val_feat, y_train_fold) if CONFIG["add_svr"] else None

    oof_tree[val_idx] = pred_tree
    oof_pair[val_idx] = pred_pair
    if pred_svr is not None:
        oof_svr[val_idx] = pred_svr

    reference_pred = np.clip(np.round(0.72 * pred_tree + 0.28 * pred_pair, 2), 0.0, 1.0)
    oof_reference[val_idx] = reference_pred

    fold_rows.append({
        "Fold": fold,
        "Tree_MAE": mean_absolute_error(y_val_fold, pred_tree),
        "Pair_MAE": mean_absolute_error(y_val_fold, pred_pair),
        "SVR_MAE": mean_absolute_error(y_val_fold, pred_svr) if pred_svr is not None else np.nan,
        "Reference_v34_MAE": mean_absolute_error(y_val_fold, reference_pred),
    })
    print(f"Fold {fold} done")

fold_df = pd.DataFrame(fold_rows)
display(fold_df)
print("개별 모델 평균 MAE:")
print(fold_df[["Tree_MAE", "Pair_MAE", "SVR_MAE"]].mean())
print()
print("### v34 고정 블렌드 기준 CV MAE (sanity check, 항상 계산됨):",
      f"{mean_absolute_error(y, oof_reference):.6f}")

if CONFIG["use_stacking"]:
    stack_inputs = [oof_tree, oof_pair] + ([oof_svr] if CONFIG["add_svr"] else [])
    stack_X = np.column_stack(stack_inputs)
    meta_model = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0])
    meta_model.fit(stack_X, y)
    oof_stacked = np.clip(np.round(meta_model.predict(stack_X), 2), 0.0, 1.0)
    print("메타모델(RidgeCV) 계수:", meta_model.coef_, "intercept:", meta_model.intercept_)
    print("### CONFIG 설정 CV MAE (RidgeCV 스태킹):", f"{mean_absolute_error(y, oof_stacked):.6f}")
else:
    current_pred = blend(oof_tree, oof_pair, oof_svr if CONFIG["add_svr"] else None)
    print("### CONFIG 설정 CV MAE (고정 블렌드):", f"{mean_absolute_error(y, current_pred):.6f}")


## 최종 제출 파일 생성

CONFIG 설정으로 CV MAE가 v34 기준(약 0.127)보다 개선된 걸 확인한 뒤에 아래 셀을 실행하세요.


In [ ]:

final_train_feat, final_test_feat = build_fold_features(raw_X, raw_X_test)

final_tree = tree_prediction(final_train_feat, final_test_feat, y)
final_pair = pair_prediction(final_train_feat, final_test_feat, y)
final_svr = svr_prediction(final_train_feat, final_test_feat, y) if CONFIG["add_svr"] else None

if CONFIG["use_stacking"]:
    stack_inputs = [final_tree, final_pair] + ([final_svr] if CONFIG["add_svr"] else [])
    final_stack_X = np.column_stack(stack_inputs)
    final_prediction = np.clip(np.round(meta_model.predict(final_stack_X), 2), 0.0, 1.0)
else:
    final_prediction = blend(final_tree, final_pair, final_svr)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_v36_ablation_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))
